# Notebook 09 — Cell–Cell Communication (Ligand–Receptor Analysis)

**Project:** RetinoblastomaAtlas  
**Input:** `data/processed/09_cellrank.h5ad`  
**Output:** `data/processed/10_communication.h5ad`, LIANA dot plots, comparison bar chart

---

## Scientific Background

Retinoblastoma progression from intraocular to extraocular disease is **not a cell-autonomous process**: tumour cells communicate with the tumour microenvironment (TME) through secreted ligands and surface receptors. Identifying which cell pairs exchange the most biologically relevant signals — and how this changes between intraocular and extraocular tumours — provides mechanistic hypotheses for invasion-driving pathways.

## Key Questions

1. Do **tumour-associated macrophages (TAMs)** in extraocular samples send more TGF-β, VEGF, or immunosuppressive (IL-10, ARG1) signals to tumour cells?
2. Do **cone precursor cells** signal back to macrophages via CSF1/IL-34 to promote M2 polarisation in the extraocular TME?
3. Which receptor-ligand interactions differ most between **Subtype 1** (non-invasive) and **Subtype 2** (invasive) tumour populations?

## Method — LIANA (Consensus L-R Scoring)

LIANA (Dimitrov et al. 2022, *Nature Communications*) aggregates scores from five L-R databases/methods:
- **CellChat** (Jin et al. 2021)
- **NicheNet** (Browaeys et al. 2020)
- **CellPhoneDB v2** (Efremova et al. 2020)
- **NATMI** (Hou et al. 2020)
- **Connectome** (Raredon et al. 2022)

Using a consensus avoids database-specific biases. Interactions are ranked by `aggregate_rank` (lower = more reliable).

## Comparison Strategy

We run L-R analysis separately on:
- **Intraocular** samples (GSE168434 + GSE249995 intraocular subset)
- **Extraocular** samples (GSE249995 extraocular subset)

And compare interaction strengths to identify invasion-associated rewiring.

**References:**  
- Dimitrov D et al. *Nat Commun* 2022. https://doi.org/10.1038/s41467-022-30755-0  
- Jin S et al. *Nat Commun* 2021. https://doi.org/10.1038/s41467-021-21246-9  
- Wan W et al. *Ophthalmology* 2025. https://doi.org/10.1016/j.ophtha.2025.01.011

## Pathways and Cell Pairs of Interest

In [ ]:
PATHWAYS_OF_INTEREST = [
    # TGF-beta signaling (invasion-promoting; Wan et al. 2025)
    "TGFB1_TGFBR1", "TGFB2_TGFBR2", "TGFB1_TGFBR2",
    # Macrophage recruitment by tumour cells
    "CSF1_CSF1R", "IL34_CSF1R",
    # Immunosuppression
    "IL10_IL10RA", "IL10_IL10RB",
    # Invasion / EMT-promoting
    "FN1_ITGB1", "FN1_ITGA5", "MMP9_ITGAV",
    # VEGF / angiogenesis
    "VEGFA_FLT1", "VEGFA_KDR",
    # Optic nerve invasion signals
    "CXCL12_CXCR4", "CXCL12_CXCR7",
    # SOX4-related signaling
    "WNT5A_FZD1", "WNT5A_ROR2",
]

CELL_TYPE_PAIRS_OF_INTEREST = [
    ("Cone_precursor", "Microglia_TAM"),
    ("Microglia_TAM",  "Cone_precursor"),
    ("Cone_precursor", "Endothelial"),
    ("Cone_precursor", "Muller_glia"),
    ("Muller_glia",    "Cone_precursor"),
]

## Setup

In [ ]:
import warnings
from pathlib import Path

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

warnings.filterwarnings("ignore")
sc.settings.verbosity = 1

ROOT     = Path("..").resolve()
IN_H5AD  = ROOT / "data" / "processed" / "09_cellrank.h5ad"
OUT_H5AD = ROOT / "data" / "processed" / "10_communication.h5ad"
FIG_DIR  = ROOT / "results" / "figures"
TAB_DIR  = ROOT / "results" / "tables"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

## Step 1 — Load CellRank-Annotated Atlas

In [ ]:
print("Step 1 — Loading CellRank-annotated atlas")
adata = sc.read_h5ad(IN_H5AD)
print(f"  {adata.n_obs:,} cells x {adata.n_vars:,} genes")
print(f"  Datasets: {adata.obs['dataset'].value_counts().to_dict()}")

## Step 2 — Split into Intraocular and Extraocular Subsets

We compare L-R interactions between intraocular and extraocular conditions. If `disease_stage` column is absent, we use `dataset` + `sample_id` as a proxy (GSE249995 samples with 'ex' in sample_id = extraocular).

In [ ]:
print("Step 2 — Splitting into intraocular and extraocular subsets")

if "disease_stage" in adata.obs.columns:
    stage_col  = "disease_stage"
    intra_mask = adata.obs[stage_col].str.contains("intraocular", case=False, na=False)
    extra_mask = adata.obs[stage_col].str.contains("extraocular", case=False, na=False)
else:
    print("  'disease_stage' not found — using dataset + sample_id as proxy")
    intra_mask = adata.obs["dataset"] == "GSE168434"
    extra_mask = (
        (adata.obs["dataset"] == "GSE249995")
        & adata.obs.get("sample_id", pd.Series("", index=adata.obs_names))
            .str.contains("ex|extraoc", case=False, na=False)
    )

adata_intra = adata[intra_mask].copy()
adata_extra = adata[extra_mask].copy()
print(f"  Intraocular: {adata_intra.n_obs:,} cells")
print(f"  Extraocular: {adata_extra.n_obs:,} cells")

## Step 3 — Run LIANA Consensus L-R Analysis

LIANA is run separately on each condition subset. Key parameters:
- `expr_prop=0.10`: gene must be expressed in ≥10% of source/target cell type
- `min_cells=20`: cell type must have ≥20 cells
- `resource_name='consensus'`: uses CellChatDB + CellPhoneDB + NATMI

The output `aggregate_rank` column (lower = more reliable) is the primary ranking metric.

**Note:** LIANA must be installed (`pip install liana`). If not installed, this step is skipped with a warning.

In [ ]:
def run_liana_on_subset(adata_sub, label, groupby="cell_type_broad", min_cells=20):
    try:
        import liana
        from liana.method import rank_aggregate
    except ImportError:
        print(f"  LIANA not installed — skipping. Install: pip install liana")
        return None

    ct_counts = adata_sub.obs[groupby].value_counts()
    valid_cts  = ct_counts[ct_counts >= min_cells].index.tolist()
    if len(valid_cts) < 2:
        print(f"  {label}: < 2 cell types with >= {min_cells} cells. Skipping.")
        return None

    adata_lr = adata_sub[adata_sub.obs[groupby].isin(valid_cts)].copy()
    print(f"  {label}: {adata_lr.n_obs:,} cells, {len(valid_cts)} cell types")

    # Use log1p of scvi_normalized for LIANA
    if "scvi_normalized" in adata_lr.layers:
        X = adata_lr.layers["scvi_normalized"]
        if sp.issparse(X):
            X = X.toarray()
        adata_lr.X = np.log1p(X)

    try:
        rank_aggregate(
            adata_lr, groupby=groupby,
            expr_prop=0.1, min_cells=min_cells,
            use_raw=False, verbose=False,
            resource_name="consensus",
        )
        results = adata_lr.uns.get("liana_res", None)
        if results is None:
            print(f"  {label}: No LIANA results found")
            return None
        results["condition"] = label
        print(f"  {label}: {len(results):,} L-R interactions computed")
        top5 = results.sort_values("aggregate_rank").head(5)
        print(f"  Top 5:\n{top5[['source','target','ligand','receptor','aggregate_rank']].to_string()}")
        return results
    except Exception as e:
        print(f"  LIANA failed for {label}: {e}")
        return None


print("Step 3 — Running LIANA consensus L-R analysis")
intra_results = run_liana_on_subset(adata_intra, "intraocular")
extra_results = run_liana_on_subset(adata_extra, "extraocular")

## Step 4 — Save Interaction Tables

In [ ]:
print("Step 4 — Saving interaction tables")

if intra_results is not None:
    intra_results.to_csv(TAB_DIR / "liana_interactions_intraocular.csv", index=False)
    print(f"  Saved intraocular -> {TAB_DIR / 'liana_interactions_intraocular.csv'}")

if extra_results is not None:
    extra_results.to_csv(TAB_DIR / "liana_interactions_extraocular.csv", index=False)
    print(f"  Saved extraocular -> {TAB_DIR / 'liana_interactions_extraocular.csv'}")

## Step 5 — Identify Extraocular-Enriched Interactions

By merging intra- and extraocular results on (source, target, ligand, receptor), we compute `delta_rank = rank_intra - rank_extra`.

**Positive delta** = lower rank in extraocular = **stronger in extraocular** = candidate invasion-promoting interaction.

In [ ]:
if intra_results is not None and extra_results is not None:
    merge_cols = ["source", "target", "ligand", "receptor"]
    merged = intra_results[merge_cols + ["aggregate_rank"]].rename(
        columns={"aggregate_rank": "rank_intra"}
    ).merge(
        extra_results[merge_cols + ["aggregate_rank"]].rename(
            columns={"aggregate_rank": "rank_extra"}
        ),
        on=merge_cols, how="outer",
    )
    merged["rank_intra"] = merged["rank_intra"].fillna(1.0)
    merged["rank_extra"] = merged["rank_extra"].fillna(1.0)
    merged["delta_rank"] = merged["rank_intra"] - merged["rank_extra"]
    merged_sorted = merged.sort_values("delta_rank", ascending=False)
    merged_sorted.to_csv(TAB_DIR / "liana_interactions_comparison.csv", index=False)

    print("Top 5 interactions enriched in extraocular:")
    print(merged_sorted.head(5)[merge_cols + ["delta_rank"]].to_string())

## Step 6 — Dot Plot of Top L-R Interactions

Dot plots show the top interactions per condition:
- **Dot size**: interaction specificity (larger = more specific)
- **Dot color**: expression magnitude

Look for TGF-β interactions (TGFB1/2 → TGFBR1/2) enriched in extraocular samples.

In [ ]:
def plot_liana_dotplot(results, label, out_path, n_top=30):
    if results is None:
        return
    try:
        from liana.pl import dotplot
    except ImportError:
        print(f"  LIANA plotting not available")
        return

    top = results.sort_values("aggregate_rank").head(n_top)
    fig, ax = plt.subplots(figsize=(14, max(8, n_top * 0.3)))
    try:
        dotplot(
            liana_res=top,
            colour="specificity_rank",
            size="magnitude_rank",
            inverse_size=True,
            inverse_colour=True,
            ax=ax, show=False,
        )
    except Exception:
        ax.scatter(
            range(len(top)),
            top["aggregate_rank"],
            s=50, c=top["aggregate_rank"], cmap="viridis_r",
        )
    ax.set_title(f"Top {n_top} L-R interactions — {label}",
                 fontsize=11, fontweight="bold")
    plt.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()

plot_liana_dotplot(intra_results, "Intraocular", FIG_DIR / "liana_dotplot_intraocular.pdf")
plot_liana_dotplot(extra_results, "Extraocular", FIG_DIR / "liana_dotplot_extraocular.pdf")

## Step 7 — Pathway Comparison Bar Chart

Direct comparison of invasion-relevant pathway interaction strengths between conditions. **TGF-β and VEGF** are expected to be higher in extraocular samples based on Wan et al. 2025.

In [ ]:
def score_interaction(results, pair):
    if results is None:
        return np.nan
    lig, rec = pair.split("_", 1)
    mask = (results["ligand"] == lig) & (results["receptor"] == rec)
    sub  = results[mask]
    if sub.empty:
        return np.nan
    return 1 - sub["aggregate_rank"].min()  # invert: higher = stronger

pathways = PATHWAYS_OF_INTEREST[:12]
intra_scores = {p: score_interaction(intra_results, p) for p in pathways}
extra_scores = {p: score_interaction(extra_results, p) for p in pathways}

df = pd.DataFrame({"intraocular": intra_scores, "extraocular": extra_scores}).dropna(how="all")

if not df.empty:
    x = np.arange(len(df))
    w = 0.35
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.bar(x - w/2, df["intraocular"].fillna(0), width=w,
           label="Intraocular", color="#4C9BE8", edgecolor="none")
    ax.bar(x + w/2, df["extraocular"].fillna(0), width=w,
           label="Extraocular", color="#E84C4C", edgecolor="none")
    ax.set_xticks(x)
    ax.set_xticklabels(df.index, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Interaction score (1 - aggregate_rank)", fontsize=10)
    ax.set_title(
        "L-R interaction strengths: intraocular vs. extraocular\n"
        "(invasion-associated pathways)",
        fontsize=11, fontweight="bold",
    )
    ax.legend(fontsize=9)
    plt.tight_layout()
    fig.savefig(FIG_DIR / "liana_top_interactions_comparison.pdf", dpi=200, bbox_inches="tight")
    plt.show()

## Step 8 — Store Results and Save Atlas

In [ ]:
if intra_results is not None:
    adata.uns["liana_intraocular"] = intra_results
if extra_results is not None:
    adata.uns["liana_extraocular"] = extra_results

print(f"Saving communication-annotated atlas -> {OUT_H5AD.name}")
adata.write_h5ad(OUT_H5AD, compression="gzip")
size_mb = OUT_H5AD.stat().st_size / 1e6
print(f"  {adata.n_obs:,} cells x {adata.n_vars:,} genes | {size_mb:.0f} MB")
print("  Added: .uns['liana_intraocular'], .uns['liana_extraocular']")
print("  Next: Run notebook 10_tgfb_pathway_scoring.ipynb")